# LoRA Fine-tuning: Llama-3.2-1B

Bu notebook Project 5'te sifirdan yazdigimiz LoRA kodunu **gercek bir LLM'e** uyguluyor.

**Pipeline:**
1. Llama-3.2-1B yukle (baz model)
2. Baseline inference -- modelin hali hazirdaki cevabi
3. `inject_lora()` -- sadece %1 parametre egitilecek
4. Alpaca dataset ile fine-tune (500 ornek, 3 epoch)
5. Fine-tuned inference -- fark ne kadar?
6. `merge_lora_weights()` -- adapteri baz modele bak
7. Merge dogrulama -- cikti identik mi?

**Runtime:** Runtime > Change runtime type > T4 GPU sec

In [ ]:
# Kurulum ve proje klonlama
!pip install -q transformers datasets accelerate huggingface_hub
!git clone https://github.com/Mertgdk/ai-lora-finetuning.git

import sys
sys.path.insert(0, '/content/ai-lora-finetuning')
print('Kurulum tamamlandi.')

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import login

from src.core.lora import inject_lora, count_parameters
from src.core.merge import merge_lora_weights

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# HuggingFace login -- token'ini buraya yaz
HF_TOKEN = 'hf_BURAYA_KENDI_TOKENINI_YAZ'
login(token=HF_TOKEN)
print('Login basarili.')

In [ ]:
# Model ve tokenizer yukle
MODEL_ID = 'meta-llama/Llama-3.2-1B'

print(f'{MODEL_ID} yukleniyor... (~2.5GB, birkaç dakika sürebilir)')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'Model yuklendi. Toplam parametre: {total_params:,}')

In [ ]:
# Baseline inference -- fine-tuning ONCESI model ne diyor?
ALPACA_TEMPLATE = (
    'Below is an instruction that describes a task. '
    'Write a response that appropriately completes the request.\n\n'
    '### Instruction:\n{instruction}\n\n### Response:\n'
)

TEST_INSTRUCTION = 'Explain what machine learning is in 3 simple sentences.'
TEST_PROMPT = ALPACA_TEMPLATE.format(instruction=TEST_INSTRUCTION)

def generate(model, tokenizer, prompt, max_new_tokens=150):
    model.eval()
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print('=== FINE-TUNING ONCESI ===')
print(f'Soru: {TEST_INSTRUCTION}\n')
baseline_answer = generate(model, tokenizer, TEST_PROMPT)
print(f'Cevap: {baseline_answer}')

In [ ]:
# LoRA inject -- q_proj ve v_proj katmanlarini hedef al
params_before = count_parameters(model)

inject_lora(model, target_modules=['q_proj', 'v_proj'], rank=8, alpha=16)

params_after = count_parameters(model)

print('=== PARAMETRE SAYILARI ===')
print(f'Toplam:          {params_after["total"]:>15,}')
print(f'Egitilecek:      {params_after["trainable"]:>15,}  ({params_after["trainable_pct"]:.4f}%)')
print(f'Dondurulmus:     {params_after["frozen"]:>15,}')
print(f'\nYani toplam modelin sadece %{params_after["trainable_pct"]:.2f}\u2019ini egitiyoruz!')

In [ ]:
# Dataset hazirla -- Alpaca'dan 500 ornek
class AlpacaDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=256):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []
        template = (
            'Below is an instruction that describes a task. '
            'Write a response that appropriately completes the request.\n\n'
            '### Instruction:\n{instruction}\n\n### Response:\n{output}'
        )
        for item in data:
            if item['output'].strip():  # bos cevaplari atla
                self.samples.append(
                    template.format(instruction=item['instruction'], output=item['output'])
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.samples[idx],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
        }

print('Alpaca dataset indiriliyor...')
raw = load_dataset('tatsu-lab/alpaca', split='train[:500]')
dataset = AlpacaDataset(raw, tokenizer, max_length=256)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

print(f'Dataset: {len(dataset)} ornek, {len(dataloader)} batch')
print(f'Ornek:\n  Instruction: {raw[0]["instruction"][:80]}...')
print(f'  Output:      {raw[0]["output"][:80]}...')

In [ ]:
# Egitim dongusu
EPOCHS = 3
LR = 2e-4

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
)

all_losses = []
epoch_losses = []

model.train()
print(f'Egitim basliyor: {EPOCHS} epoch, lr={LR}, batch=4')
print('-' * 50)

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    n_batches = 0

    for i, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        # padding tokenlarini loss'dan cikar
        labels = input_ids.clone()
        labels[labels == tokenizer.pad_token_id] = -100

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        all_losses.append(loss.item())
        n_batches += 1

        if (i + 1) % 20 == 0:
            print(f'  Epoch {epoch+1} | Batch {i+1}/{len(dataloader)} | Loss: {loss.item():.4f}')

    avg = epoch_loss / n_batches
    epoch_losses.append(avg)
    print(f'Epoch {epoch+1}/{EPOCHS} tamamlandi -- Ort. Loss: {avg:.4f}')
    print('-' * 50)

print(f'\nLoss: {epoch_losses[0]:.4f} -> {epoch_losses[-1]:.4f} ({100*(epoch_losses[0]-epoch_losses[-1])/epoch_losses[0]:.1f}% azaldi)')

In [ ]:
# Loss grafigi
plt.figure(figsize=(10, 4))
plt.plot(all_losses, alpha=0.4, label='Batch loss')

# epoch ortalamalarini ciz
batches_per_epoch = len(dataloader)
for i, el in enumerate(epoch_losses):
    x = (i + 0.5) * batches_per_epoch
    plt.axhline(el, xmin=(i * batches_per_epoch) / len(all_losses),
                xmax=((i + 1) * batches_per_epoch) / len(all_losses),
                color='red', linewidth=2, label=f'Epoch {i+1} avg: {el:.4f}' if i == 0 else f'Epoch {i+1}: {el:.4f}')

plt.xlabel('Batch')
plt.ylabel('Loss')
plt.title('LoRA Fine-tuning Loss (Llama-3.2-1B)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Fine-tuning SONRASI inference -- fark var mi?
print('=== FINE-TUNING ONCESI ===')
print(f'Cevap: {baseline_answer}\n')

print('=== FINE-TUNING SONRASI (merge oncesi) ===')
print(f'Soru: {TEST_INSTRUCTION}\n')
finetuned_answer = generate(model, tokenizer, TEST_PROMPT)
print(f'Cevap: {finetuned_answer}')

In [ ]:
# Merge -- LoRA adapter'i baz modele bak, wrapper'i kaldir
# Once merge oncesi ciktiyi kaydet
test_input = tokenizer(TEST_PROMPT, return_tensors='pt').to(device)
model.eval()

with torch.no_grad():
    logits_before_merge = model(**test_input).logits.clone()

merge_lora_weights(model)

with torch.no_grad():
    logits_after_merge = model(**test_input).logits

max_diff = (logits_before_merge - logits_after_merge).abs().max().item()

params_merged = count_parameters(model)

print('=== MERGE SONUCLARI ===')
print(f'Max logit farki: {max_diff:.2e}')
print(f'Merge {"BASARILI" if max_diff < 1e-2 else "BASARISIZ"} (esik: 1e-2)')
print(f'\nMerge sonrasi param sayisi: {params_merged["total"]:,} (lora ekleri gitti)')
print(f'Trainable: {params_merged["trainable_pct"]:.2f}% (100% - hepsi egitilir artik)')

In [ ]:
# Final karsilastirma
print('=== KARSILASTIRMA ===')
print(f'Soru: {TEST_INSTRUCTION}\n')
print('ONCESI:')
print(baseline_answer)
print('\nSONRASI (fine-tuned + merged):')
merged_answer = generate(model, tokenizer, TEST_PROMPT)
print(merged_answer)

print('\n=== OZET ===')
print(f'Egitilen parametre:    {params_after["trainable_pct"]:.4f}% ({params_after["trainable"]:,} / {params_after["total"]:,})')
print(f'Loss dususu:           {epoch_losses[0]:.4f} -> {epoch_losses[-1]:.4f}')
print(f'Merge dogrulama:       max diff = {max_diff:.2e}')
print(f'\nProject 5 kodumuz gercek bir 1B LLM uzerinde calisli!')

In [ ]:
# (Opsiyonel) Farkli sorular dene
test_instructions = [
    'Write a haiku about artificial intelligence.',
    'What are the benefits of exercise?',
    'Explain gravity to a 5-year-old.',
]

for instr in test_instructions:
    prompt = ALPACA_TEMPLATE.format(instruction=instr)
    answer = generate(model, tokenizer, prompt, max_new_tokens=100)
    print(f'Soru:  {instr}')
    print(f'Cevap: {answer}')
    print('-' * 60)